In [ ]:
# Kernel Restarts
!pip install --upgrade ipywidgets --quiet
!pip install --upgrade transformers --quiet
!pip install --upgrade pip --quiet
!pip install -q ftfy regex tqdm torch torchvision git+https://github.com/openai/CLIP.git

# !pip install -U transformers==4.54.1 --quiet
!pip uninstall -y tensorflow --quiet && !pip install tensorflow-cpu --quiet

print("Done installing! Restarting kernel...")
os.kill(os.getpid(), 9)

  Preparing metadata (setup.py) ... done
/bin/bash: line 1: !pip: command not found


# Imports

In [8]:
# ===================================================================== #
# ==================== STEP 1: IMPORTS for CLIP ======================= #
# ===================================================================== #

# Install OpenAI CLIP (if not already)
# !pip install -q git+https://github.com/openai/CLIP.git

# Imports
import os
import pandas as pd
import torch
import clip
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm


# ===================================================================== #
# ========================== IMPORTS for T5 =========================== #
# ===================================================================== #



import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import json
import os


from tqdm.notebook import tqdm
import random

print("="*50)
print("All imports complete")

All imports complete


In [ ]:
# multimodal_5fold_cv.py
import os
import random
import pickle
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    T5EncoderModel,
    CLIPProcessor,
    CLIPModel,
    get_scheduler,
)

print("="*50)
print("All imports complete")

In [3]:
SENTIMENTS_TO_LABELS = {"negative": 0, "neutral": 1, "positive": 2}
SENTIMENTS_TO_LABELS = {0: "negative", 1: "neutral", 2: "positive"}

In [4]:
# ----------------------------
# User-editable paths & names
# ----------------------------
CSV_PATH = "/kaggle/input/musait/MUSAIT__all_info_uniques.csv"   # <<-- change to your CSV path
OUTPUT_DIR = "./multimodal_cv_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# df = pd.read_csv(CSV_PATH)
# df.head(2)

In [6]:
# df['filepath'] = df['filepath'].str.replace('dataset/basket', 'basket_unique')
# df.to_csv(CSV_PATH.split("/")[-1])

In [10]:
# CSV expected columns mapping (edit if different)
CSV_COLS = {
    "image": "filepath",   # column containing image file paths
    "text":  "OCR",         # column containing OCR/text
    "label": "sentiment",        # column containing labels (0,1,2) OR text labels
}


# ----------------------------
# Hyperparams (taken from T5 notebook)
# ----------------------------
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 2e-4
BATCH_SIZE = 16
EPOCHS = 25
MAX_LENGTH = 128
NUM_FOLDS = 5
NUM_CLASSES = 3
SAVE_MODELS = True

# reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("="*50)
print("Seed set")

Seed set


In [11]:
# ----------------------------
# Dataset
# ----------------------------
from PIL import Image

class MultimodalDataset(Dataset):
    def __init__(self, df, tokenizer, clip_processor, max_length=128, cols=CSV_COLS):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.clip_processor = clip_processor
        self.max_length = max_length
        self.cols = cols

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row[self.cols["filepath"]]
        text = str(row[self.cols["OCR"]])
        label = SENTIMENTS_TO_LABELS[row[self.cols["sentiment"]]]

        # load image
        image = Image.open(img_path).convert("RGB")

        # text encoding (T5 tokenizer)
        text_inputs = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # CLIP image processing
        image_inputs = self.clip_processor(images=image, return_tensors="pt")

        # squeeze tensors (remove batch dim)
        ti = {k: v.squeeze(0) for k, v in text_inputs.items()}
        ii = {k: v.squeeze(0) for k, v in image_inputs.items()}

        return {
            "input_ids": ti["input_ids"],
            "attention_mask": ti["attention_mask"],
            "pixel_values": ii["pixel_values"],
            "label": torch.tensor(int(label), dtype=torch.long),
        }

In [12]:
# ----------------------------
# Model wrappers (lightweight heads)
# ----------------------------
class T5TextClassifier(nn.Module):
    def __init__(self, model_name="t5-small", num_classes=3):
        super().__init__()
        # Use T5 encoder for classification (consistent with your T5 notebook)
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.d_model
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # mean pool last_hidden_state (batch, seq_len, hidden)
        last = out.last_hidden_state  # (B, L, H)
        pooled = last.mean(dim=1)     # (B, H)
        logits = self.classifier(pooled)
        return logits

class CLIPImageClassifier(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", num_classes=3):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        # clip.get_image_features returns features with projection size clip.config.projection_dim
        # but clip model typically exposes .visual_projection (or we can use clip.visual.encoder)
        # we'll use clip.get_image_features followed by a linear projection
        img_feature_dim = self.clip.visual_projection.out_features if hasattr(self.clip, "visual_projection") else self.clip.config.projection_dim
        self.classifier = nn.Linear(img_feature_dim, num_classes)

    def forward(self, pixel_values):
        # clip.get_image_features handles normalization & forward through visual tower
        # If batch size = 1, ensure shapes consistent
        img_feats = self.clip.get_image_features(pixel_values=pixel_values)
        logits = self.classifier(img_feats)
        return logits

In [13]:
# ----------------------------
# Utility functions
# ----------------------------
def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        optimizer.zero_grad()
        labels = batch["label"].to(device)
        with autocast():
            if isinstance(model, T5TextClassifier):
                logits = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                )
            else:
                logits = model(pixel_values=batch["pixel_values"].to(device))

            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits.detach().cpu(), dim=1).numpy()
        correct += (preds == labels.cpu().numpy()).sum()
        total += labels.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc

In [14]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch["label"].to(device)
            if isinstance(model, T5TextClassifier):
                logits = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                )
            else:
                logits = model(pixel_values=batch["pixel_values"].to(device))

            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.detach().cpu().numpy())

            running_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            correct += (preds == labels.detach().cpu().numpy()).sum()
            total += labels.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    return avg_loss, acc, all_probs, all_labels

In [ ]:
SENTIMENTS_TO_LABELS[CSV_COLS["sentiment"]]

In [ ]:
# ----------------------------
# Read CSV and prepare labels
# ----------------------------
df = pd.read_csv(CSV_PATH)

# Normalize column names if required; ensure label column numeric
label_col = CSV_COLS["label"]
if df[label_col].dtype == object and SENTIMENTS_TO_LABELS is not None:
    df[label_col] = df[label_col].map(SENTIMENTS_TO_LABELS)
    if df[label_col].isnull().any():
        raise ValueError("Some labels could not be mapped. Check TEXT_LABEL_TO_INT mapping.")

In [ ]:
# ----------------------------
# Tokenizers / processors
# ----------------------------
t5_name = "t5-small"  # same as your T5 notebook baseline
clip_name = "openai/clip-vit-base-patch32"

tokenizer = AutoTokenizer.from_pretrained(t5_name)
clip_processor = CLIPProcessor.from_pretrained(clip_name)

# ----------------------------
# Cross-validation setup
# ----------------------------
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
X = df[[CSV_COLS["image"], CSV_COLS["text"]]]
y = df[CSV_COLS["label"]].values

fold_results = []
all_fold_histories = {}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n=== Fold {fold}/{NUM_FOLDS} ===")

    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df  = df.iloc[test_idx].reset_index(drop=True)

    # Optionally create a small validation split out of train_df (we'll do 90/10)
    val_frac = 0.1
    val_count = max(1, int(len(train_df) * val_frac))
    val_df = train_df.sample(n=val_count, random_state=SEED)
    train_df = train_df.drop(val_df.index).reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    # Datasets & loaders
    train_dataset = MultimodalDataset(train_df, tokenizer, clip_processor, max_length=MAX_LENGTH)
    val_dataset   = MultimodalDataset(val_df, tokenizer, clip_processor, max_length=MAX_LENGTH)
    test_dataset  = MultimodalDataset(test_df, tokenizer, clip_processor, max_length=MAX_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # Instantiate models for this fold
    text_model = T5TextClassifier(model_name=t5_name, num_classes=NUM_CLASSES).to(DEVICE)
    image_model = CLIPImageClassifier(model_name=clip_name, num_classes=NUM_CLASSES).to(DEVICE)

    # optimizer grouping & AdamW
    def setup_optimizer(model, lr):
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": 0.01,
            },
            {
                "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
            },
        ]
        return AdamW(optimizer_grouped_parameters, lr=lr)

    opt_text = setup_optimizer(text_model, LEARNING_RATE)
    opt_image = setup_optimizer(image_model, LEARNING_RATE)

    # Schedulers: linear decay as in reference
    # compute num_training_steps for each model: epochs * (len(train_loader))
    num_training_steps_text = EPOCHS * len(train_loader)
    num_training_steps_image = EPOCHS * len(train_loader)

    scheduler_text = get_scheduler(
        name="linear",
        optimizer=opt_text,
        num_warmup_steps=0,
        num_training_steps=num_training_steps_text,
    )
    scheduler_image = get_scheduler(
        name="linear",
        optimizer=opt_image,
        num_warmup_steps=0,
        num_training_steps=num_training_steps_image,
    )

    criterion = nn.CrossEntropyLoss()

    # scaler for AMP
    scaler_text = GradScaler()
    scaler_image = GradScaler()

    # For history logging
    history = {
        "text": {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []},
        "image": {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []},
        "fused_test_acc": None
    }

    # ----------------------------
    # Training loops for this fold
    # ----------------------------
    for epoch in range(1, EPOCHS + 1):
        # text train epoch
        t_loss, t_acc = train_one_epoch(text_model, train_loader, opt_text, scheduler_text, criterion, DEVICE, scaler_text)
        vt_loss, vt_acc, _, _ = validate(text_model, val_loader, criterion, DEVICE)

        # image train epoch
        i_loss, i_acc = train_one_epoch(image_model, train_loader, opt_image, scheduler_image, criterion, DEVICE, scaler_image)
        vi_loss, vi_acc, _, _ = validate(image_model, val_loader, criterion, DEVICE)

        history["text"]["train_loss"].append(t_loss)
        history["text"]["train_acc"].append(t_acc)
        history["text"]["val_loss"].append(vt_loss)
        history["text"]["val_acc"].append(vt_acc)

        history["image"]["train_loss"].append(i_loss)
        history["image"]["train_acc"].append(i_acc)
        history["image"]["val_loss"].append(vi_loss)
        history["image"]["val_acc"].append(vi_acc)

        print(f"Fold {fold} Epoch {epoch}/{EPOCHS} | "
              f"T: train_loss={t_loss:.4f}, train_acc={t_acc:.4f}, val_loss={vt_loss:.4f}, val_acc={vt_acc:.4f} || "
              f"I: train_loss={i_loss:.4f}, train_acc={i_acc:.4f}, val_loss={vi_loss:.4f}, val_acc={vi_acc:.4f}")

    # ----------------------------
    # Evaluate on test set and do probability fusion
    # ----------------------------
    # Get text probs
    _, _, text_probs_test, text_labels_test = validate(text_model, test_loader, criterion, DEVICE)
    _, _, image_probs_test, image_labels_test = validate(image_model, test_loader, criterion, DEVICE)

    # Sanity: labels must match
    assert (text_labels_test == image_labels_test).all()

    fused_probs = 0.5 * text_probs_test + 0.5 * image_probs_test
    fused_preds = np.argmax(fused_probs, axis=1)
    fused_acc = accuracy_score(text_labels_test, fused_preds)

    print(f"Fold {fold} fused test accuracy: {fused_acc:.4f}")

    history["fused_test_acc"] = float(fused_acc)
    fold_results.append(fused_acc)
    all_fold_histories[f"fold_{fold}"] = history

    # save history pickle
    hist_path = os.path.join(OUTPUT_DIR, f"history_fold_{fold}.pkl")
    with open(hist_path, "wb") as f:
        pickle.dump(history, f)

    # save models (optional)
    if SAVE_MODELS:
        torch.save(text_model.state_dict(), os.path.join(OUTPUT_DIR, f"text_model_fold_{fold}.pt"))
        torch.save(image_model.state_dict(), os.path.join(OUTPUT_DIR, f"image_model_fold_{fold}.pt"))

    # ----------------------------
    # Plot learning curves and save
    # ----------------------------
    def plot_and_save(history_part, title, save_path):
        epochs_range = range(1, len(history_part["train_loss"]) + 1)
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        plt.plot(epochs_range, history_part["train_loss"], label="train_loss")
        plt.plot(epochs_range, history_part["val_loss"], label="val_loss")
        plt.title(title + " - Loss")
        plt.xlabel("Epoch")
        plt.legend()
        plt.subplot(1,2,2)
        plt.plot(epochs_range, history_part["train_acc"], label="train_acc")
        plt.plot(epochs_range, history_part["val_acc"], label="val_acc")
        plt.title(title + " - Accuracy")
        plt.xlabel("Epoch")
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()

    plot_and_save(history["text"], f"Fold{fold}_T5", os.path.join(OUTPUT_DIR, f"learning_curve_fold_{fold}_t5.png"))
    plot_and_save(history["image"], f"Fold{fold}_CLIP", os.path.join(OUTPUT_DIR, f"learning_curve_fold_{fold}_clip.png"))

# ----------------------------
# Summary across folds
# ----------------------------
fold_results = np.array(fold_results)
mean_acc = float(fold_results.mean())
std_acc = float(fold_results.std(ddof=1))

print(f"\n=== Cross-validation summary over {NUM_FOLDS} folds ===")
for i, acc in enumerate(fold_results, start=1):
    print(f"Fold {i}: fused_test_acc = {acc:.4f}")
print(f"Mean fused accuracy = {mean_acc:.4f}")
print(f"Std dev fused accuracy = {std_acc:.4f}")

# Save final summary & histories
summary = {
    "fold_accuracies": fold_results.tolist(),
    "mean_accuracy": mean_acc,
    "std_accuracy": std_acc,
    "all_histories": all_fold_histories,
    "params": {
        "LEARNING_RATE": LEARNING_RATE,
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "MAX_LENGTH": MAX_LENGTH
    }
}
with open(os.path.join(OUTPUT_DIR, "cv_summary.pkl"), "wb") as f:
    pickle.dump(summary, f)

# plot the fused accuracies bar plot
plt.figure(figsize=(8,4))
plt.bar(range(1, NUM_FOLDS+1), fold_results)
plt.errorbar(NUM_FOLDS+1, mean_acc, yerr=std_acc, fmt='o', label='mean±std')
plt.title("Fused test accuracies per fold")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.xticks(range(1, NUM_FOLDS+1))
plt.savefig(os.path.join(OUTPUT_DIR, "fused_accuracy_per_fold.png"))
plt.close()

print(f"All results and plots saved to: {os.path.abspath(OUTPUT_DIR)}")